# 为什么要将多份年报整理到一个excel中？
企业年报通常是一家公司一年一次的总结：经营情况、收入利润、行业背景……  

但如果我们需要分析 很多公司、很多年份 的年报，就必须把它们的内容汇总成一个表格（例如 Excel），这样才能：  

• 进行文本分析（如分词、提取关键词）  

• 做对比（不同公司、不同年份）  

• 做可视化（趋势图、柱状图等  

可以把这件事想象成：
你有一堆书（年报文件），每本书里都有公司名字、年份、正文。你的任务是把这些信息抄到一个大表格里。
将多个文件汇总到一个Excel中，解决步骤 :

|Step|功能|解决方案|
|:---|:---|:---|
|1|获取文件路径列表|pathlib库|
|2|尝试对任意文件进行读取，获取A相关信息|open函数、字符串操作获取相关信息|
|3|将2中得到相关信息存入csv文件|csv库|
|5|for循环，对所有的文件进行2、3操作| for循环，注意代码层次|

## Step 1：获取所有年报文件的路径
（就像先把书架上所有书的名字列出来）  

在计算机中，一个文件的“地址”叫做 路径（path）。  

我们需要做的第一件事就是：  

告诉 Python：这些年报都放在哪个文件夹？请帮我把文件列表拿出来。  

在 Python 中通常用 pathlib 来做这件事。

## Step 2：读取其中一个文件并提取我们想要的内容
（像是翻开一本书，把关键信息抄到笔记本）  

一个年报文件一般是 .txt 或 .pdf 转成的文本。  

我们想从中提取：  

• 公司名称（name）  

• 年份（year）  

• 年报正文（text）  

提取方法通常是用：  

• open() 读取文本  

• 用字符串操作找出 “公司名称” 或 “2017 年度” 等字样

#### 注意：关于 encoding
不同文档保存文字的方式可能不同，就像中文和英文书不是同一种纸张。  

在 Python 中读取中文文件时，很多同学会遇到乱码。  

如果读取报错，可尝试 encoding="gbk" 或加入 errors="ignore" 让 Python 跳过读不出来的字符。

## Step 3：把提取出的信息放进一个 CSV / Excel
（像把笔记整理成表格）  

当我们成功读出：  

• name  

• year  

• text  

就可以把它们放进一个表格里。  

这一步常用：  

• csv 库（写入 csv 文件）  

• 或者 pandas.DataFrame.to_excel()

## Step 4：使用 for 循环，让电脑自动处理所有文件
（让电脑重复同样的动作，而不是手动打开一百次）  

逻辑：  

“对文件夹里的每一个文件：打开 → 读取 → 提取 → 写入表格”  

这就是 for 循环。  

电脑能比人快地多，一秒几十个文件不是问题。

In [3]:
# 步骤一  新建 CSV 表格（写入表头
from pathlib import Path    # 用来处理文件路径（比字符串更安全）
import csv                  # 用来写 CSV 文件

# 打开一个 CSV 文件（如果没有会自动创建）
csvf = open(
    'datanew/output/reports.csv',   # 输出文件路径
    'a+',                   # a+：追加写入模式（append），适合逐条写
    encoding='utf-8'        # 以 UTF-8 编码写入，避免中文乱码
)

# 定义 CSV 的列名（字段）
fieldnames = ['name', 'year', 'text']

# 创建一个“字典写入器”（接受字典并写成 CSV 格式）
writer = csv.DictWriter(csvf, fieldnames=fieldnames)

# 写入表头（第一行）
writer.writeheader()

16

步骤二 批量获取所有 txt 文件路径  <br>
如果文件不是txt格式的，要转换下格式：<br>
方法 1：手动把 PDF 转成 txt<br>
方法 2：用 Python 先把 PDF 转成 txt 再读<br>
• pdfplumber<br>
• PyMuPDF (fitz)<br>
• pdfminer<br>
来转换成文本。<br>

In [4]:
# 步骤二 获取所有txt文件路径
# 获取 data/txts 文件夹中所有 .txt 文件的路径
files = Path().cwd() \
              .joinpath('datanew', 'txts') \
              .glob('*.txt')

# files 是一个生成器，每次循环返回 1 个 txt 文件路径
# 1.Path().cwd()找到当前工作目录（当前 Notebook 所在的文件夹）——当前站的位置
# 2.joinpath('data','txts')进入 data/txts 这个子文件夹 —— 走进小房间
# 3..glob('*.txt')找到里面所有以 .txt 结尾的文件 —— 把小房间以.txt结尾的书都拿出来

In [5]:
# 步骤三：从单个 txt 文件中提取 name / year / text
for file in files:         # 遍历每一个 txt 文件路径
    
    file = str(file)       # 转成字符串路径，方便 split 操作

    # 从路径中取出文件名，去掉后缀 .txt
    # '/xxx/yyy/四川长虹2017.txt' → '四川长虹2017'
    rawinfo = file.split('/')[-1][:-4]  
    #把路径当成“按 / 分割的拼音串”切开，取列表最后一个元素，去掉最后 4 个字符（即去掉后缀 .txt）

    # 文件名最后 4 位是年份，例如 '2017'
    year = rawinfo[-4:]  #取最后四个字符

    # 年份前面的部分就是公司名称，例如 '四川长虹'
    name = rawinfo[:-4] #从开头到倒数第 4 位之前，不包含倒数第 4 位

    # 读取 txt 文件内容（假设保存为 gbk 编码）
    text = open(file, encoding='gbk').read()

In [7]:
# 步骤四：写入 CSV 
# 整理成字典格式，键名要与 fieldnames 对应
data = {
     'name': name,
     'year': year,
     'text': text
    }  #把步骤3采集的信息放入data

# 写入一行到 CSV 文件
writer.writerow(data)

# 所有文件处理完毕后，关闭 CSV 文件
csvf.close()

In [8]:
# 查看写入的结果
import pandas as pd

df = pd.read_csv('output/reports.csv', encoding='utf-8')
df.head()   # 显示前 5 行

,name,year,text
0,四川长虹,2017,2017 年，面对复杂多变的外部环境和多重叠加的困难挑战，公司聚焦用户与产品，强化消费洞...
1,江苏吴中,2014,2014 年，正值公司成立二十周年，上市十五周年，在董事会带领下，公司经营管理团队与全体...
2,联美控股,2017,"报告期内，公司实现营业收入 2,376,375,380.44 元，同比增长 16.24%，营..."
3,华海药业,2016,第三节\t公司业务概要\n\n一、 报告期内公司所从事的主要业务、经营模式及行业情况说明\n...
4,江泉实业,2014,报告期内，全球经济形势复杂多变、复苏进程缓慢；我国宏观经济进入增速放缓、结构调整加剧的新...


In [9]:
# 查看长度
len(df)

500

In [10]:
#年份
df['year'].unique()

array([2017, 2014, 2016, 2015, 2013])

In [11]:
df['name'].unique()

array(['四川长虹', '江苏吴中', '联美控股', '华海药业', '江泉实业', '杭萧钢构', '德宏股份', '亚通股份',
       '广汇能源', '厦门钨业', '亿利洁能', '卧龙地产', '维维股份', '中国交建', '中远海控', '国检集团',
       '北京城乡', '浙能电力', '海立股份', '常熟银行', '亚翔集成', '中电电机', '广深铁路', '隆鑫通用',
       '金晶科技', '华仪电气', '钱江生化', '黑牡丹股份', '长电科技', '东软集团', '东百集团', '中远海发',
       '凤竹纺织', '晶方科技', '中海油服', '三棵树', '江苏金融', '陕西煤业', '佳力图', '继峰股份',
       '旗滨集团', '诚意药业', '丽岛新材', '华域汽车', '上海九百', '烽火通信', '嘉澳环保', '西部矿业',
       '道森股份', '百利电气', '彩虹股份', '上港集团', '天地源', '金桥信息', '天药股份', '同济科技',
       '杉杉股份', '大众公用', '长春燃气', '沧州大化', '梅雁吉祥', '鲁商置业', '美凯龙', '开开实业',
       '同达创业', '睿能科技', '南京化纤', '桃李面包', '鸣志电器', '老百姓', '中国铁建', '两面针',
       '宁波富邦', '欧亚集团', '新世界', '麦迪科技', '法拉电子', '华银电力', '兴业证券', '华夏幸福',
       '澳柯玛', '青海华鼎', '中国中铁', '汇通能源', '永创智能', '金枫酒业', '杭齿前进', '绿庭投资',
       '林洋能源', '亚泰集团', '江中药业', '海峡环保', '通化东宝', '人民网', '宁波高发', '新钢股份',
       '洛阳钼业', '华脉科技', '汇丽 B', '大千生态', '中国中冶', '东宏股份', '合力科技', '永安行',
       '新华锦', '福建水泥', '风神股份', '建设银行', '天安新材', '醋化股份', '科达洁能', '金隅集团',
       '石油化工', '

In [12]:
#公司有多少家
df['name'].nunique()

220

In [13]:
for name in df['name'].unique():
    ndf = df[df['name']==name]
    if len(ndf)>4:
        print(name)

江苏吴中
联美控股
华海药业
江泉实业
亚通股份
广汇能源
厦门钨业
亿利洁能
卧龙地产
维维股份
北京城乡
浙能电力
海立股份
金晶科技
华仪电气
黑牡丹股份
长电科技
东软集团
东百集团
彩虹股份
上港集团
天地源
同济科技
杉杉股份
大众公用
长春燃气
沧州大化
鲁商置业
开开实业
同达创业
两面针
欧亚集团
新世界
法拉电子
华夏幸福
澳柯玛
青海华鼎
汇通能源
金枫酒业
亚泰集团
重庆港九
熊猫金控
东风汽车
钱江水利
长江通信
大连圣亚
福田汽车
丰华股份
腾达建设
通葡股份
光明地产
宝钢股份
江西铜业
恒瑞医药
南京新百
西昌电力
物产中大
南京医药
东贝电器
银座股份
华鑫股份
江西长运
贵州茅台
华联综超
航天通信
汇丽建材
中恒集团
东风科技
京能电力
首旅酒店


In [14]:
df[df['name']=='首旅酒店']

,name,year,text
413,首旅酒店,2013,\n 面对 2013 年严峻的经济环境和复杂的市场发展态势，公司全面实施《酒店品牌发展战略...
445,首旅酒店,2017,"重组、整合、创新、发展贯穿着首旅如家的 2017 年，公司遵循“向存量和中端要效益,向..."
452,首旅酒店,2016,公司酒店集团 2016 年遵循“改革创新、转型发展、提质增效”核心发展目标，认真研讨酒...
486,首旅酒店,2014,2014 年，面对全国饭店业收入、利润仍处低位运行的局势，公司董事会积极实施“资本+品\n\...
490,首旅酒店,2015,\n2015 年公司初步建立起首旅酒店集团的品牌谱系、形成较为合理的成员酒店区域布局、建立\...
